# Wafer Map Defect Classification Training Pipeline


In [1]:
import os
import pickle
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# Configure GPU Memory Growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled")
    except RuntimeError as e:
        print(e)
        
os.makedirs(r"d:\assignment college\VLSI_PROJECT\models", exist_ok=True)


GPU Memory Growth Enabled


## Phase 3: Handcrafted Feature (MFE) FNN Base Classifier


In [2]:
print("Loading extracted MFE data and labels...")
with open(r"d:\assignment college\VLSI_PROJECT\data\X_MFE.pkl", 'rb') as f:
    X_train_mfe, X_test_mfe = pickle.load(f)
with open(r"d:\assignment college\VLSI_PROJECT\data\y.pkl", 'rb') as f:
    y_train, y_test = pickle.load(f)
    
print(f"X_train_mfe shape: {X_train_mfe.shape}, y_train shape: {y_train.shape}")
print(f"X_test_mfe shape: {X_test_mfe.shape}, y_test shape: {y_test.shape}")

# Cast to float32 to save RAM
X_train_mfe = X_train_mfe.astype(np.float32)
X_test_mfe = X_test_mfe.astype(np.float32)

# Standardize the features (zero mean, unit variance) as mentioned in paper
mean = np.mean(X_train_mfe, axis=0)
std = np.std(X_train_mfe, axis=0)
std[std == 0] = 1 # prevent division by zero

X_train_mfe_scaled = (X_train_mfe - mean) / std
X_test_mfe_scaled = (X_test_mfe - mean) / std

# Calculate class weights for imbalance handling
classes = np.unique(y_train)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {c: w for c, w in zip(classes, class_weights_array)}
print(f"Computed class weights: {class_weight_dict}")

y_train_onehot = tf.keras.utils.to_categorical(y_train, num_classes=9)
y_test_onehot = tf.keras.utils.to_categorical(y_test, num_classes=9)


Loading extracted MFE data and labels...
X_train_mfe shape: (162946, 59), y_train shape: (162946,)
X_test_mfe shape: (10000, 59), y_test shape: (10000,)
Computed class weights: {0: 4.474817377931565, 1: 34.61780327172296, 2: 3.703234017408695, 3: 1.9852095516569201, 4: 5.348629574922041, 5: 22.18763616557734, 6: 16.107750098853302, 7: 129.32222222222222, 8: 0.1303435570945992}


In [12]:
# Build FNN Architecture (Table 3 from paper)
mfe_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(59,)),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(128, activation='tanh'),
    tf.keras.layers.Dense(9, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
mfe_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

print("Training MFE-FNN Base Classifier...")
mfe_model.fit(
    X_train_mfe_scaled, y_train_onehot,
    validation_split=0.2,
    epochs=1000,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)


Training MFE-FNN Base Classifier...
Epoch 1/1000
4074/4074 [==============================] - 27s 6ms/step - loss: 0.7953 - accuracy: 0.7283 - val_loss: 0.5535 - val_accuracy: 0.8464
Epoch 2/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.4573 - accuracy: 0.8744 - val_loss: 0.3875 - val_accuracy: 0.8911
Epoch 3/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.3953 - accuracy: 0.8908 - val_loss: 0.4093 - val_accuracy: 0.8790
Epoch 4/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.3649 - accuracy: 0.8935 - val_loss: 0.3496 - val_accuracy: 0.8964
Epoch 5/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.3399 - accuracy: 0.8988 - val_loss: 0.3444 - val_accuracy: 0.8957
Epoch 6/1000
4074/4074 [==============================] - 26s 6ms/step - loss: 0.3220 - accuracy: 0.9010 - val_loss: 0.3478 - val_accuracy: 0.8960
Epoch 7/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.3

In [5]:
print("Evaluating MFE FNN on Test Set...")
y_pred_prob_mfe = mfe_model.predict(X_test_mfe_scaled, batch_size=64)
y_pred_mfe = np.argmax(y_pred_prob_mfe, axis=1)

report_mfe = classification_report(y_test, y_pred_mfe, digits=4)
print("Classification Report (MFE+FNN):")
print(report_mfe)

f1_macro_mfe = f1_score(y_test, y_pred_mfe, average='macro')
print(f"F1 Macro: {f1_macro_mfe:.4f}")

# Generate predictions on train and test for the stacking ensemble
y_train_pred_prob_mfe = mfe_model.predict(X_train_mfe_scaled, batch_size=64)

mfe_model.save(r"d:\assignment college\VLSI_PROJECT\models\mfe_fnn_model.keras")

with open(r"d:\assignment college\VLSI_PROJECT\data\mfe_fnn_outputs.pkl", 'wb') as f:
    pickle.dump({
        'train_prob': y_train_pred_prob_mfe,
        'test_prob': y_pred_prob_mfe,
        'y_train': y_train,
        'y_test': y_test,
        'report': report_mfe,
        'f1_macro': f1_macro_mfe
    }, f)
print("Phase 3 MFE-FNN Completed!")


Evaluating MFE FNN on Test Set...


NameError: name 'mfe_model' is not defined

## Phase 4: CNN Base Classifier


In [6]:
print("Loading extracted CNN data and labels...")
with open(r"d:\assignment college\VLSI_PROJECT\data\X_CNN.pkl", 'rb') as f:
    X_train_cnn, X_test_cnn = pickle.load(f)
    
print(f"X_train_cnn shape: {X_train_cnn.shape}")

# Cast to float32, expand dims, and scale to save RAM
X_train_cnn = np.expand_dims(X_train_cnn.astype(np.float32), axis=-1)
X_test_cnn = np.expand_dims(X_test_cnn.astype(np.float32), axis=-1)

X_train_cnn = (X_train_cnn - 0.5) * 2.0
X_test_cnn = (X_test_cnn - 0.5) * 2.0

print(f"X_train_cnn pre-processed shape: {X_train_cnn.shape}")


Loading extracted CNN data and labels...
X_train_cnn shape: (162946, 64, 64, 1)
X_train_cnn pre-processed shape: (162946, 64, 64, 1, 1)


In [7]:
# Build VGGNet Architecture (Table 4 from paper)
cnn_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(64, 64, 1)),
    tf.keras.layers.Lambda(lambda x: tf.repeat(x, 3, axis=-1)),
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.Conv2D(512, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(9, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
cnn_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

print("Training CNN Base Classifier...")
cnn_model.fit(
    X_train_cnn, y_train_onehot,
    validation_split=0.2,
    epochs=1000,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stopping],
    verbose=1
)


Training CNN Base Classifier...
Epoch 1/1000
4074/4074 [==============================] - 307s 72ms/step - loss: 1.5582 - accuracy: 0.4753 - val_loss: 0.7772 - val_accuracy: 0.8986
Epoch 2/1000
4074/4074 [==============================] - 288s 71ms/step - loss: 0.8104 - accuracy: 0.7761 - val_loss: 0.8069 - val_accuracy: 0.7837
Epoch 3/1000
4074/4074 [==============================] - 288s 71ms/step - loss: 0.5623 - accuracy: 0.8251 - val_loss: 0.1949 - val_accuracy: 0.9417
Epoch 4/1000
4074/4074 [==============================] - 288s 71ms/step - loss: 0.4338 - accuracy: 0.8513 - val_loss: 0.7549 - val_accuracy: 0.7369
Epoch 5/1000
4074/4074 [==============================] - 288s 71ms/step - loss: 0.3813 - accuracy: 0.8763 - val_loss: 0.2826 - val_accuracy: 0.9133
Epoch 6/1000
4074/4074 [==============================] - 288s 71ms/step - loss: 0.2844 - accuracy: 0.8957 - val_loss: 0.2144 - val_accuracy: 0.9300
Epoch 7/1000
4074/4074 [==============================] - 288s 71ms/step -

In [9]:
print("Evaluating CNN on Test Set...")
y_pred_prob_cnn = cnn_model.predict(X_test_cnn, batch_size=64)
y_pred_cnn = np.argmax(y_pred_prob_cnn, axis=1)

report_cnn = classification_report(y_test, y_pred_cnn, digits=4)
print("Classification Report (CNN):")
print(report_cnn)

f1_macro_cnn = f1_score(y_test, y_pred_cnn, average='macro')
print(f"F1 Macro: {f1_macro_cnn:.4f}")

# Generate predictions on train and test for the stacking ensemble
y_train_pred_prob_cnn = cnn_model.predict(X_train_cnn, batch_size=64)

cnn_model.save(r"d:\assignment college\VLSI_PROJECT\models\cnn_model.keras")

with open(r"d:\assignment college\VLSI_PROJECT\data\cnn_outputs.pkl", 'wb') as f:
    pickle.dump({
        'train_prob': y_train_pred_prob_cnn,
        'test_prob': y_pred_prob_cnn,
        'y_train': y_train,
        'y_test': y_test,
        'report': report_cnn,
        'f1_macro': f1_macro_cnn
    }, f)
print("Phase 4 CNN Completed!")


Evaluating CNN on Test Set...
157/157 [==============================] - 5s 32ms/step
Classification Report (CNN):
              precision    recall  f1-score   support

           0     0.8566    0.9637    0.9070       248
           1     0.7750    0.9688    0.8611        32
           2     0.7417    0.8233    0.7804       300
           3     0.9803    0.9750    0.9776       560
           4     0.6532    0.7788    0.7105       208
           5     0.7458    0.8800    0.8073        50
           6     0.6023    0.7681    0.6752        69
           7     1.0000    0.8889    0.9412         9
           8     0.9943    0.9784    0.9863      8524

    accuracy                         0.9670     10000
   macro avg     0.8166    0.8917    0.8496     10000
weighted avg     0.9708    0.9670    0.9684     10000

F1 Macro: 0.8496


InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [10]:
# 1. Save the model IMMEDIATELY before it crashes again
cnn_model.save(r"d:/assignment college/VLSI_PROJECT/models/cnn_model.keras")
print("Model saved safely!")

# 2. Force Windows and TensorFlow to dump unused VRAM
import gc
gc.collect()

# 3. Predict using a tiny batch size so the 6GB VRAM doesn't overflow
print("Running predictions with safe batch size...")
y_train_pred_prob_cnn = cnn_model.predict(X_train_cnn, batch_size=16)

# 4. Save the outputs for Phase 5 Stacking
import pickle
with open(r"d:/assignment college/VLSI_PROJECT/data/cnn_outputs.pkl", 'wb') as f:
    pickle.dump({
        'train_prob': y_train_pred_prob_cnn,
        'test_prob': y_pred_prob_cnn,
        'y_train': y_train,
        'y_test': y_test,
        'report': report_cnn,
        'f1_macro': f1_macro_cnn
    }, f)
print("Phase 4 CNN completely saved and finished!")


Model saved safely!
Running predictions with safe batch size...


InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [12]:
import numpy as np
import pickle

print("Running manual Python loop to bypass TensorFlow's VRAM memory greed...")
batch_size = 64
num_samples = len(X_train_cnn)
predictions = []

# Spoon-feed the GPU exactly one batch at a time
for i in range(0, num_samples, batch_size):
    batch = X_train_cnn[i:i+batch_size]
    pred_batch = cnn_model.predict(batch, verbose=0)
    predictions.append(pred_batch)
    
    if i % 10000 < batch_size:
        print(f"Processed {i}/{num_samples} samples...")

# Stitch all the batches back together
y_train_pred_prob_cnn = np.concatenate(predictions, axis=0)

print("Saving outputs...")
with open(r"d:/assignment college/VLSI_PROJECT/data/cnn_outputs.pkl", 'wb') as f:
    pickle.dump({
        'train_prob': y_train_pred_prob_cnn,
        'test_prob': y_pred_prob_cnn,
        'y_train': y_train,
        'y_test': y_test,
        'report': report_cnn,
        'f1_macro': f1_macro_cnn
    }, f)
print("Phase 4 CNN completely saved and finished! You may proceed to Phase 5!")


Running manual Python loop to bypass TensorFlow's VRAM memory greed...
Processed 0/162946 samples...
Processed 10048/162946 samples...
Processed 20032/162946 samples...
Processed 30016/162946 samples...
Processed 40000/162946 samples...
Processed 50048/162946 samples...
Processed 60032/162946 samples...
Processed 70016/162946 samples...
Processed 80000/162946 samples...
Processed 90048/162946 samples...
Processed 100032/162946 samples...
Processed 110016/162946 samples...
Processed 120000/162946 samples...
Processed 130048/162946 samples...
Processed 140032/162946 samples...
Processed 150016/162946 samples...
Processed 160000/162946 samples...
Saving outputs...
Phase 4 CNN completely saved and finished! You may proceed to Phase 5!


In [13]:
import pickle
with open(r"d:/assignment college/VLSI_PROJECT/data/mfe_fnn_outputs.pkl", 'rb') as f:
    mfe_data = pickle.load(f)
y_train_pred_prob_mfe = mfe_data['train_prob']
y_pred_prob_mfe = mfe_data['test_prob']


## Phase 5: Stacking Ensemble


In [14]:
print("Preparing Stacking Data...")
# We use the predicted probabilities from MFE and CNN as input features for the meta-learner
X_train_concat = np.concatenate([y_train_pred_prob_mfe, y_train_pred_prob_cnn], axis=1)
X_test_concat = np.concatenate([y_pred_prob_mfe, y_pred_prob_cnn], axis=1)

print(f"X_train_concat shape: {X_train_concat.shape}")
print(f"X_test_concat shape: {X_test_concat.shape}")

# FNN Meta-Learner
stacking_model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(18,)),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(9, activation='softmax')
])
stacking_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

print("Training Stacking Meta-Learner...")
stacking_model.fit(
    X_train_concat, y_train_onehot,
    validation_split=0.2,
    epochs=1000,
    batch_size=32,
    class_weight=class_weight_dict,  # fix: unweighted meta-learner early-stops at ~23 epochs and drops rare classes (Near-full F1=0)
    callbacks=[early_stopping],
    verbose=1
)


Preparing Stacking Data...
X_train_concat shape: (162946, 18)
X_test_concat shape: (10000, 18)
Training Stacking Meta-Learner...
Epoch 1/1000
4074/4074 [==============================] - 26s 6ms/step - loss: 0.8668 - accuracy: 0.7284 - val_loss: 0.2917 - val_accuracy: 0.8708
Epoch 2/1000
4074/4074 [==============================] - 24s 6ms/step - loss: 0.1573 - accuracy: 0.9532 - val_loss: 0.1391 - val_accuracy: 0.9677
Epoch 3/1000
4074/4074 [==============================] - 26s 6ms/step - loss: 0.0484 - accuracy: 0.9937 - val_loss: 0.1040 - val_accuracy: 0.9728
Epoch 4/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.0236 - accuracy: 0.9946 - val_loss: 0.1092 - val_accuracy: 0.9733
Epoch 5/1000
4074/4074 [==============================] - 25s 6ms/step - loss: 0.0186 - accuracy: 0.9948 - val_loss: 0.1176 - val_accuracy: 0.9728
Epoch 6/1000
4074/4074 [==============================] - 26s 6ms/step - loss: 0.0168 - accuracy: 0.9949 - val_loss: 0.1251 - val_accura

In [16]:
f1_macro_mfe = mfe_data['f1_macro']


In [17]:
print("Evaluating Stacking Ensemble on Test Set...")
y_pred_prob_stack = stacking_model.predict(X_test_concat)
y_pred_stack = np.argmax(y_pred_prob_stack, axis=1)

report_stack = classification_report(y_test, y_pred_stack, digits=4)
print("Classification Report (Stacking Ensemble):")
print(report_stack)

f1_macro_stack = f1_score(y_test, y_pred_stack, average='macro')
print(f"F1 Macro: {f1_macro_stack:.4f}")

stacking_model.save(r"d:\assignment college\VLSI_PROJECT\models\stacking_model.keras")

with open(r"d:\assignment college\VLSI_PROJECT\data\stacking_outputs.pkl", 'wb') as f:
    pickle.dump({
        'test_prob': y_pred_prob_stack,
        'y_test': y_test,
        'report': report_stack,
        'f1_macro': f1_macro_stack,
        'mfe_f1_macro': f1_macro_mfe,
        'cnn_f1_macro': f1_macro_cnn,
    }, f)
print("Phase 5 Stacking Completed!")


Evaluating Stacking Ensemble on Test Set...
313/313 [==============================] - 1s 2ms/step
Classification Report (Stacking Ensemble):
              precision    recall  f1-score   support

           0     0.9105    0.9435    0.9267       248
           1     0.9000    0.8438    0.8710        32
           2     0.8425    0.7667    0.8028       300
           3     0.9891    0.9679    0.9783       560
           4     0.8434    0.6731    0.7487       208
           5     0.8113    0.8600    0.8350        50
           6     0.8889    0.5797    0.7018        69
           7     0.0000    0.0000    0.0000         9
           8     0.9824    0.9944    0.9883      8524

    accuracy                         0.9732     10000
   macro avg     0.7964    0.7366    0.7614     10000
weighted avg     0.9712    0.9732    0.9717     10000

F1 Macro: 0.7614
Phase 5 Stacking Completed!


d:\Anaconda3\envs\btp_lstm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Anaconda3\envs\btp_lstm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Anaconda3\envs\btp_lstm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [19]:
import numpy as np
y_pred_mfe = np.argmax(y_pred_prob_mfe, axis=1)


## Phase 6: Model Comparison


In [20]:
target_names = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Random', 'Scratch', 'Near-full', 'none']

# Calculate per-class metrics
p_mfe, r_mfe, f1_mfe_class, _ = precision_recall_fscore_support(y_test, y_pred_mfe)
p_cnn, r_cnn, f1_cnn_class, _ = precision_recall_fscore_support(y_test, y_pred_cnn)
p_stk, r_stk, f1_stk_class, _ = precision_recall_fscore_support(y_test, y_pred_stack)

markdown_output = "# Final Model Comparison Results\\n\\n"
markdown_output += "## Per-Class F1 Score Comparison\\n\\n"
markdown_output += "| Class | MFE+FNN | CNN | Stacking Ensemble |\\n"
markdown_output += "|-------|---------|-----|-------------------|\\n"

for i, name in enumerate(target_names):
    markdown_output += f"| {name} | {f1_mfe_class[i]:.4f} | {f1_cnn_class[i]:.4f} | {f1_stk_class[i]:.4f} |\\n"
    
markdown_output += f"| **Macro Avg** | **{f1_macro_mfe:.4f}** | **{f1_macro_cnn:.4f}** | **{f1_macro_stack:.4f}** |\\n"

print(markdown_output)

with open(r"d:\assignment college\VLSI_PROJECT\results_comparison.md", 'w') as f:
    f.write(markdown_output)
    
print("Phase 6 comparison saved to results_comparison.md!")


# Final Model Comparison Results\n\n## Per-Class F1 Score Comparison\n\n| Class | MFE+FNN | CNN | Stacking Ensemble |\n|-------|---------|-----|-------------------|\n| Center | 0.8973 | 0.9070 | 0.9267 |\n| Donut | 0.7826 | 0.8611 | 0.8710 |\n| Edge-Loc | 0.7265 | 0.7804 | 0.8028 |\n| Edge-Ring | 0.9529 | 0.9776 | 0.9783 |\n| Loc | 0.6507 | 0.7105 | 0.7487 |\n| Random | 0.8381 | 0.8073 | 0.8350 |\n| Scratch | 0.5618 | 0.6752 | 0.7018 |\n| Near-full | 0.9412 | 0.9412 | 0.0000 |\n| none | 0.9811 | 0.9863 | 0.9883 |\n| **Macro Avg** | **0.8147** | **0.8496** | **0.7614** |\n
Phase 6 comparison saved to results_comparison.md!


d:\Anaconda3\envs\btp_lstm_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
